# Cell 1

In [ ]:
# =============================================================================
# Thesis — GeoAI Café Site Selection · Milan
# Phase 2 · Notebook 5 · AHP Scoring
# =============================================================================
#
# Purpose:
#   Apply the Analytic Hierarchy Process (AHP) to produce a structured,
#   expert-grounded suitability score for every viable H3 cell.
#
#   The AHP score is one of two independent suitability assessments:
#     - AHP score:    expert judgment encoded in pairwise comparison matrix
#     - RF score:     data-driven, learned from real café locations (NB6)
#   Their comparison is the core academic contribution of this thesis.
#
# AHP structure — two levels:
#
#   Level 1 — Four criteria clusters with global weights
#     C1: Accessibility       (metro, transit, centrality)
#     C2: Demand Potential    (population, night light)
#     C3: Urban Context       (retail, tourism, POI diversity, pedestrian street)
#     C4: Competition         (café density, saturation)
#
#   Level 2 — Sub-weights within each cluster
#     Applied to normalized feature values before cluster aggregation
#
# Process:
#   1. Define pairwise comparison matrix (4x4) across criteria clusters
#   2. Compute weight vector via eigenvector method
#   3. Verify Consistency Ratio (CR) < 0.10
#   4. Define sub-weights within each cluster
#   5. Compute cluster scores as weighted sum of normalized features
#   6. Compute final AHP suitability score as weighted sum of cluster scores
#   7. Save AHP scores to Gold layer
#
# Inputs:
#   - gold_features_normalized.geojson
#
# Outputs:
#   - gold_ahp_scores.geojson
#   - gold_ahp_log.txt
#
# CRS: EPSG:4326 (GeoJSON standard — analysis already complete)
# =============================================================================

# Cell 2

In [ ]:
!pip install geopandas -q
!pip install numpy -q
!pip install pandas -q

import os
import datetime
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', module='osmnx')
os.environ['OGR_GEOJSON_MAX_OBJ_SIZE'] = '0'

import numpy as np
import pandas as pd
import geopandas as gpd

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Thesis'
BRONZE_PATH  = os.path.join(PROJECT_ROOT, 'Bronze')
SILVER_PATH  = os.path.join(PROJECT_ROOT, 'Silver')
GOLD_PATH    = os.path.join(PROJECT_ROOT, 'Gold')

print(f"Notebook started: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Notebook started: 2026-04-26 12:03


# Cell 3

In [ ]:
print("Loading normalized feature table from Gold layer...")

features = gpd.read_file(
    os.path.join(GOLD_PATH, 'gold_features_normalized.geojson')
)

print(f"  Cells loaded:  {len(features)}")
print(f"  Columns:       {list(features.columns)}")
print()

# Verify all normalized feature columns are present
# NOTE on feature counts:
#   15 GDF columns total    : 13 model features + viable_overlap_ratio + label
#   13 RF model features    : all features passed to Random Forest
#   10 AHP-scored inputs    : 9 _norm columns + pedestrian_street (binary)
#   RF-only (not AHP-scored): local_cafe_density_norm
#
# AHP input breakdown by cluster:
#   Accessibility (3): metro_count_norm, transit_density_norm, network_centrality_norm
#   Demand        (2): pop_density_norm, night_light_norm
#   Urban Context (3): retail_density_norm, tourist_poi_count_norm, poi_diversity_norm
#   Urban Context (+1 binary): pedestrian_street
#   Competition   (1): competitor_saturation_inv_norm
#   Total         = 9 _norm columns + 1 binary = 10 AHP inputs

# --- Hard stop: 9 AHP-scored _norm columns ---
# These are required for AHP cluster score computation.
# Absence of any of these columns indicates a NB4 regression and must halt.
ahp_norm_cols = [
    # Accessibility (3)
    'metro_count_norm',
    'transit_density_norm',
    'network_centrality_norm',
    # Demand Potential (2) — Phase 2 canonical 2-feature structure
    'pop_density_norm',
    'night_light_norm',
    # Urban Context (3)
    'retail_density_norm',
    'tourist_poi_count_norm',
    'poi_diversity_norm',
    # Competition (1)
    'competitor_saturation_inv_norm', # canonical name after inversion + norm
]

missing_ahp = [c for c in ahp_norm_cols if c not in features.columns]
assert not missing_ahp, (
    f"Missing AHP-scored normalized columns in gold_features_normalized.geojson: "
    f"{missing_ahp}. Re-run NB4."
)
print(f"  All 9 AHP-scored _norm columns present")

# --- Hard stop: pedestrian_street binary column ---
# Used in Urban Context cluster (URBAN_SUB_WEIGHTS). Binary [0,1] — not _norm.
assert 'pedestrian_street' in features.columns, (
    "Missing AHP input column: pedestrian_street. "
    "This binary flag is required for the Urban Context cluster score. Re-run NB4."
)
print(f"  pedestrian_street present (binary AHP input, Urban Context cluster)")
print(f"  Total AHP inputs verified: 10 (9 _norm + 1 binary)")

# --- Soft warning: RF-only column ---
# local_cafe_density_norm is not used in any AHP cluster.
# Its absence does not affect AHP scoring but may indicate a NB4 change
# that should be reviewed before running NB6.
if 'local_cafe_density_norm' not in features.columns:
    print(f"  WARNING — local_cafe_density_norm not found. "
          f"Not used in AHP scoring, but required by NB6 (Random Forest). "
          f"Review NB4 output before proceeding to NB6.")
else:
    print(f"  local_cafe_density_norm present (RF-only — not used in AHP)")

print()
print("Feature count summary:")
print(f"  15 GDF columns total    : 13 model features + viable_overlap_ratio + label")
print(f"  13 RF model features    : all AHP inputs + local_cafe_density_norm + office/uni")
print(f"  10 AHP-scored inputs    : 9 _norm columns + pedestrian_street (binary)")
print(f"  RF-only (not AHP-scored): local_cafe_density_norm, office_density_norm, university_proximity_norm")

Loading normalized feature table from Gold layer...
  Cells loaded:  971
  Columns:       ['h3_id', 'label', 'cafe_count', 'viable_overlap_ratio', 'pedestrian_street', 'centroid_x', 'centroid_y', 'metro_count_norm', 'transit_density_norm', 'network_centrality_norm', 'pop_density_norm', 'office_density_norm', 'university_proximity_norm', 'night_light_norm', 'retail_density_norm', 'tourist_poi_count_norm', 'poi_diversity_norm', 'local_cafe_density_norm', 'competitor_saturation_inv_norm', 'geometry']

  All 9 AHP-scored _norm columns present
  pedestrian_street present (binary AHP input, Urban Context cluster)
  Total AHP inputs verified: 10 (9 _norm + 1 binary)
  local_cafe_density_norm present (RF-only — not used in AHP)

Feature count summary:
  15 GDF columns total    : 13 model features + viable_overlap_ratio + label
  13 RF model features    : all AHP inputs + local_cafe_density_norm + office/uni
  10 AHP-scored inputs    : 9 _norm columns + pedestrian_street (binary)
  RF-only (not

# Cell 4

In [ ]:
# AHP computation functions
# These implement the standard Saaty AHP method:
#   1. Normalize the pairwise comparison matrix by column sums
#   2. Compute the weight vector as row averages of normalized matrix
#   3. Compute lambda_max (principal eigenvalue)
#   4. Compute Consistency Index (CI) and Consistency Ratio (CR)
#   5. Reject if CR >= 0.10

# Saaty Random Index table
# Used to compute the Consistency Ratio
# RI[n] is the average random consistency index for a matrix of size n
RI = {1: 0.00, 2: 0.00, 3: 0.58, 4: 0.90, 5: 1.12,
      6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49}

def compute_ahp_weights(matrix, name="matrix"):
    """
    Computes AHP weight vector, consistency diagnostics, and consistency
    ratio from a pairwise comparison matrix.

    Parameters
    ----------
    matrix : np.ndarray — square pairwise comparison matrix
    name   : str — label for reporting

    Returns
    -------
    weights    : np.ndarray — normalized weight vector (sums to 1)
    cr         : float — consistency ratio (must be < 0.10)
    lambda_max : float — principal eigenvalue
    ci         : float — consistency index
    ri         : float — Saaty random index for n
    """
    n = matrix.shape[0]

    # Step 1 — Normalize matrix by column sums
    col_sums    = matrix.sum(axis=0)
    matrix_norm = matrix / col_sums

    # Step 2 — Weight vector = row averages of normalized matrix
    weights = matrix_norm.mean(axis=1)

    # Step 3 — Compute lambda_max (principal eigenvalue)
    weighted_sum  = matrix @ weights
    lambda_values = weighted_sum / weights
    lambda_max    = lambda_values.mean()

    # Step 4 — Consistency Index
    ci = (lambda_max - n) / (n - 1)

    # Step 5 — Consistency Ratio
    ri = RI.get(n, 1.49)
    cr = ci / ri if ri > 0 else 0.0

    # Report
    print(f"\n  AHP results for: {name}")
    print(f"    n (criteria):   {n}")
    print(f"    lambda_max:     {lambda_max:.6f}")
    print(f"    CI:             {ci:.6f}")
    print(f"    RI:             {ri}")
    print(f"    CR:             {cr:.6f}  "
          f"{'[VALID — CR < 0.10]' if cr < 0.10 else '[INVALID — CR >= 0.10 — REVISE MATRIX]'}")
    print(f"    Weights:        {np.round(weights, 4)}")

    if cr >= 0.10:
        raise ValueError(
            f"Consistency Ratio {cr:.4f} >= 0.10 for {name}. "
            f"The pairwise comparison matrix is inconsistent. "
            f"Revise the matrix values and re-run."
        )

    return weights, cr, lambda_max, ci, ri

print("AHP helper functions defined")
print()
print("Saaty Random Index table:")
for k, v in RI.items():
    print(f"  n={k}: RI={v}")

AHP helper functions defined

Saaty Random Index table:
  n=1: RI=0.0
  n=2: RI=0.0
  n=3: RI=0.58
  n=4: RI=0.9
  n=5: RI=1.12
  n=6: RI=1.24
  n=7: RI=1.32
  n=8: RI=1.41
  n=9: RI=1.45
  n=10: RI=1.49


# Cell 5

In [ ]:
# =============================================================================
# PAIRWISE COMPARISON MATRIX — Level 1 (Criteria Clusters)
# =============================================================================
# Revised weights (Phase 2 final): Access≈37.6%, Demand≈30.0%, Urban≈25.3%, Compet≈7.0%
#
# Rationale: Urban Context raised to reflect Milan's walkable, morphology-driven
# retail geography (Arias Muñoz et al. 2018). Accessibility reduced from 55.8%
# because transit access is near-uniform across viable H3 cells at this
# resolution, limiting its discriminatory power. Competition raised above 5.7%
# analytical invisibility threshold.
#
#              C1    C2    C3    C4
# C1 Access   [1,    1,    2,    5  ]
# C2 Demand   [1,    1,    1,    4  ]
# C3 Urban    [1/2,  1,    1,    4  ]
# C4 Compet   [1/5,  1/4,  1/4,  1  ]
# =============================================================================

print("Defining Level 1 pairwise comparison matrix...")
print()
print("Criteria clusters:")
print("  C1: Accessibility")
print("  C2: Demand Potential")
print("  C3: Urban Context")
print("  C4: Competition")
print()

# Define the 4x4 pairwise comparison matrix
ahp_matrix_l1 = np.array([
    [1,      1,      2,      5   ],   # C1 Accessibility
    [1,      1,      1,      4   ],   # C2 Demand Potential
    [1/2,    1,      1,      4   ],   # C3 Urban Context
    [1/5,    1/4,    1/4,    1   ],   # C4 Competition
])

cluster_names = ['Accessibility', 'Demand Potential',
                 'Urban Context', 'Competition']

print("Pairwise comparison matrix (Level 1):")
df_matrix = pd.DataFrame(
    ahp_matrix_l1.round(3),
    index=cluster_names,
    columns=cluster_names
)
print(df_matrix.to_string())
print()

# Compute weights and verify consistency
# compute_ahp_weights returns lambda_max, CI, and RI alongside weights and cr,
# so the diagnostics block below draws from identical arithmetic — no duplication.
global_weights, global_cr, global_lambda_max, global_ci, global_ri = compute_ahp_weights(
    ahp_matrix_l1, "Level 1 — Criteria Clusters"
)

# =============================================================================
# AHP consistency diagnostics — sourced directly from compute_ahp_weights()
# return values. Guarantees the printed values and global_cr are derived from
# identical arithmetic. Verifies RI=0.90 is used for n=4 (not n=3's RI=0.58).
# =============================================================================
print()
print("=" * 55)
print("AHP CONSISTENCY DIAGNOSTICS — Level 1 (4×4 matrix)")
print("=" * 55)
print(f"  n (criteria):         {ahp_matrix_l1.shape[0]}")
print(f"  λ_max:                {global_lambda_max:.6f}")
print(f"  CI = (λ_max-n)/(n-1): {global_ci:.6f}")
print(f"  RI (n=4, Saaty):      {global_ri}  ← must be 0.90 for a 4×4 matrix")
print(f"  CR = CI / RI:         {global_cr:.6f}  "
      f"({'VALID — CR < 0.10' if global_cr < 0.10 else 'INVALID — CR >= 0.10'})")
print("=" * 55)
print()

# Store named weights for clarity
W = {
    'accessibility':    global_weights[0],
    'demand_potential': global_weights[1],
    'urban_context':    global_weights[2],
    'competition':      global_weights[3],
}

print("Global cluster weights:")
for name, weight in W.items():
    print(f"  {name:<20}: {weight:.4f} ({weight*100:.1f}%)")
print(f"  {'TOTAL':<20}: {sum(W.values()):.4f}")

Defining Level 1 pairwise comparison matrix...

Criteria clusters:
  C1: Accessibility
  C2: Demand Potential
  C3: Urban Context
  C4: Competition

Pairwise comparison matrix (Level 1):
                  Accessibility  Demand Potential  Urban Context  Competition
Accessibility               1.0              1.00           2.00          5.0
Demand Potential            1.0              1.00           1.00          4.0
Urban Context               0.5              1.00           1.00          4.0
Competition                 0.2              0.25           0.25          1.0


  AHP results for: Level 1 — Criteria Clusters
    n (criteria):   4
    lambda_max:     4.047279
    CI:             0.015760
    RI:             0.9
    CR:             0.017511  [VALID — CR < 0.10]
    Weights:        [0.3764 0.2998 0.2535 0.0703]

AHP CONSISTENCY DIAGNOSTICS — Level 1 (4×4 matrix)
  n (criteria):         4
  λ_max:                4.047279
  CI = (λ_max-n)/(n-1): 0.015760
  RI (n=4, Saaty):      0.

# Cell 6

In [ ]:
# =============================================================================
# SUB-WEIGHTS — Level 2 (Features within each cluster)
# =============================================================================
#
# Sub-weights determine how features are combined within each cluster.
# Sub-weights within each cluster must sum to 1.
# These are directly assigned expert weights (not derived from pairwise
# matrices), following the deliberate simplification precedent in
# Shaikh et al. 2020 — documented explicitly in the thesis methodology.
#
# IMPORTANT: local_cafe_density is excluded from all AHP clusters.
# It is retained as an RF-only feature. Including it in AHP would be
# circular — it directly proxies the target variable (café presence).
# competitor_saturation_inv_norm (inverted, normalized) is the AHP
# competition proxy, not local_cafe_density.
#
# Phase 2 Demand cluster — 2-feature structure (ISTAT 2021 upgrade):
#   office_density and university_proximity were dropped from the AHP
#   Demand cluster after the upgrade to ISTAT 2021 data (master_handover_final.md
#   Section 2). Both features are retained as RF model features in NB6.
#
# Sub-weight constants are defined here as named dicts and used in both
# scoring computation and log generation, ensuring the log reflects
# the values actually used in computation.
# =============================================================================

print("Defining sub-weights within each criteria cluster...")
print()

# C1 — Accessibility sub-weights (REVISED)
# Surface transit (trams/buses) is the dominant day-to-day mobility layer in
# Milan's inner ring. Network centrality is the best structural proxy for
# pedestrian throughput. Metro downweighted: near-uniform coverage across
# viable cells limits discriminatory power.
ACCESS_SUB_WEIGHTS = {
    'metro_count_norm':          0.25,
    'transit_density_norm':      0.40,
    'network_centrality_norm':   0.35,
}

# C2 — Demand Potential sub-weights (REVISED)
# Night light majority-weighted: proxies daytime economic activity and
# non-residential footfall (workers, tourists) that drive aperitivo/coffee
# demand. Population density downweighted: Milan café economy is not
# solely residential-catchment driven.
DEMAND_SUB_WEIGHTS = {
    'pop_density_norm':  0.45,
    'night_light_norm':  0.55,
}

# C3 — Urban Context sub-weights (REVISED)
# Retail density remains primary (ground-floor continuity = commercial street).
# POI diversity and pedestrian street raised to equal weight — both signal
# mixed-use dwell-time environments. Tourist POI raised to 0.20: Milan receives
# 10M+ annual visitors concentrated in specific districts.
URBAN_SUB_WEIGHTS = {
    'retail_density_norm':       0.30,
    'poi_diversity_norm':        0.25,
    'pedestrian_street':         0.25,
    'tourist_poi_count_norm':    0.20,
}

# C4 — Competition sub-weights (unchanged — sole AHP competition signal)
COMPETITION_SUB_WEIGHTS = {
    'competitor_saturation_inv_norm': 1.00,
}

sub_weights_accessibility = ACCESS_SUB_WEIGHTS
sub_weights_demand        = DEMAND_SUB_WEIGHTS
sub_weights_urban         = URBAN_SUB_WEIGHTS
sub_weights_competition   = COMPETITION_SUB_WEIGHTS

# Verify sub-weights sum to 1.0 within each cluster
all_sub_weights = {
    'Accessibility':    sub_weights_accessibility,
    'Demand Potential': sub_weights_demand,
    'Urban Context':    sub_weights_urban,
    'Competition':      sub_weights_competition,
}

print("Sub-weight verification (each cluster must sum to 1.0):")
all_valid = True
for cluster_name, sw in all_sub_weights.items():
    total = sum(sw.values())
    valid = abs(total - 1.0) < 1e-6
    if not valid:
        all_valid = False
    print(f"  {cluster_name:<20}: {total:.4f} "
          f"{'[OK]' if valid else '[ERROR — must sum to 1.0]'}")
    for feat, w in sw.items():
        print(f"    {feat:<45}: {w:.2f}")

print()
if all_valid:
    print("All sub-weights valid")
else:
    raise ValueError(
        "Sub-weights do not sum to 1.0 in one or more clusters. "
        "Correct the values above before proceeding."
    )

Defining sub-weights within each criteria cluster...

Sub-weight verification (each cluster must sum to 1.0):
  Accessibility       : 1.0000 [OK]
    metro_count_norm                             : 0.25
    transit_density_norm                         : 0.40
    network_centrality_norm                      : 0.35
  Demand Potential    : 1.0000 [OK]
    pop_density_norm                             : 0.45
    night_light_norm                             : 0.55
  Urban Context       : 1.0000 [OK]
    retail_density_norm                          : 0.30
    poi_diversity_norm                           : 0.25
    pedestrian_street                            : 0.25
    tourist_poi_count_norm                       : 0.20
  Competition         : 1.0000 [OK]
    competitor_saturation_inv_norm               : 1.00

All sub-weights valid


# Cell 7

In [ ]:
# Compute the score for each cluster as the weighted sum
# of the normalized features within that cluster
#
# Cluster score = sum(feature_value * sub_weight)
# for all features in the cluster
#
# Result: one score per cluster per H3 cell, all in [0, 1]

print("Computing cluster scores...")
print()

def compute_cluster_score(df, sub_weights, cluster_name):
    """
    Computes the weighted sum of features within a cluster.

    Parameters
    ----------
    df          : DataFrame containing normalized feature columns
    sub_weights : dict mapping column name to sub-weight
    cluster_name: str for reporting

    Returns
    -------
    pd.Series of cluster scores in [0, 1]
    """
    score = pd.Series(0.0, index=df.index)
    for col, weight in sub_weights.items():
        if col in df.columns:
            assert not df[col].isna().any(), f"NaN found in normalized feature column: {col}"
            score += df[col] * weight
        else:
            raise KeyError(f"Column '{col}' not found in DataFrame — cannot compute cluster score.")
    return score

# Compute each cluster score
features['score_accessibility'] = compute_cluster_score(
    features, sub_weights_accessibility, 'Accessibility'
)
features['score_demand'] = compute_cluster_score(
    features, sub_weights_demand, 'Demand Potential'
)
features['score_urban'] = compute_cluster_score(
    features, sub_weights_urban, 'Urban Context'
)
features['score_competition'] = compute_cluster_score(
    features, sub_weights_competition, 'Competition'
)

# Report cluster score distributions
cluster_score_cols = [
    'score_accessibility', 'score_demand',
    'score_urban', 'score_competition'
]

print("Cluster score distributions (all should be in [0, 1]):")
print(features[cluster_score_cols].describe().round(4).to_string())

Computing cluster scores...

Cluster score distributions (all should be in [0, 1]):
       score_accessibility  score_demand  score_urban  score_competition
count             971.0000      971.0000     971.0000           971.0000
mean                0.3195        0.6532       0.4496             0.8963
std                 0.1238        0.1646       0.2188             0.0963
min                 0.0000        0.1062       0.0000             0.0000
25%                 0.2416        0.5769       0.2897             0.8503
50%                 0.3064        0.7208       0.4371             0.9114
75%                 0.4008        0.7680       0.5712             0.9655
max                 0.8226        0.9226       1.0328             1.0000


# Cell 8

In [ ]:
# Compute the final AHP suitability score as the weighted sum
# of the four cluster scores using the global Level 1 weights
#
# AHP_score = W_access * score_access
#           + W_demand * score_demand
#           + W_urban  * score_urban
#           + W_compet * score_compet
#
# Result: one composite score per H3 cell in [0, 1]
# Higher score = more suitable location for a café

print("Computing final AHP suitability score...")
print()

features['ahp_score'] = (
    W['accessibility']    * features['score_accessibility'] +
    W['demand_potential'] * features['score_demand']        +
    W['urban_context']    * features['score_urban']         +
    W['competition']      * features['score_competition']
)

# Verify score range
print(f"  AHP score range: "
      f"{features['ahp_score'].min():.4f} — "
      f"{features['ahp_score'].max():.4f}")
print(f"  AHP score mean:  {features['ahp_score'].mean():.4f}")
print(f"  AHP score std:   {features['ahp_score'].std():.4f}")
print()

# Score distribution by label class
# Positive cells (label=1) should generally score higher than
# negative cells (label=0) if AHP weights are well-calibrated
pos_scores = features[features['label'] == 1]['ahp_score']
neg_scores = features[features['label'] == 0]['ahp_score']

print("AHP score by class label:")
print(f"  Positive cells (label=1): "
      f"mean={pos_scores.mean():.4f}, "
      f"median={pos_scores.median():.4f}")
print(f"  Negative cells (label=0): "
      f"mean={neg_scores.mean():.4f}, "
      f"median={neg_scores.median():.4f}")
print()

# Rank cells by AHP score
features['ahp_rank'] = features['ahp_score'].rank(
    ascending=False, method='min'
).astype(int)

print("Top 10 cells by AHP score:")
top10_cols = ['h3_id', 'ahp_score', 'ahp_rank', 'label',
              'cafe_count', 'score_accessibility',
              'score_demand', 'score_urban', 'score_competition']
top10_present = [c for c in top10_cols if c in features.columns]
print(features.nsmallest(10, 'ahp_rank')[top10_present].to_string(index=False))

Computing final AHP suitability score...

  AHP score range: 0.1043 — 0.8016
  AHP score mean:  0.4931
  AHP score std:   0.1163

AHP score by class label:
  Positive cells (label=1): mean=0.5770, median=0.5685
  Negative cells (label=0): mean=0.4468, median=0.4494

Top 10 cells by AHP score:
          h3_id  ahp_score  ahp_rank  label  cafe_count  score_accessibility  score_demand  score_urban  score_competition
891f99cd9dbffff   0.801562         1      1           4             0.822579      0.780925     0.810419           0.745098
891f99cdd47ffff   0.790899         2      1          10             0.568153      0.852651     1.032814           0.848101
891f99cdd4fffff   0.788078         3      1          15             0.617561      0.783450     1.027639           0.857143
891f99cdd0fffff   0.782204         4      0           1             0.639736      0.775967     0.954351           0.950980
891f99cdd4bffff   0.779976         5      1           7             0.574931      0.816906 

# Cell 9

In [ ]:
# Validation checks on the AHP score
# These checks verify the score is analytically meaningful
# before saving to Gold and using in the WebGIS

print("Running AHP score validation checks...")
print()

checks_passed = True

# Check 1 — Score range
min_score = features['ahp_score'].min()
max_score = features['ahp_score'].max()
check1 = (min_score >= 0.0) and (max_score <= 1.0)
print(f"  [{'PASS' if check1 else 'FAIL'}] Score range in [0,1]: "
      f"min={min_score:.4f}, max={max_score:.4f}")
if not check1:
    checks_passed = False

# Check 2 — Positive cells score higher than negative on average
pos_mean = pos_scores.mean()
neg_mean = neg_scores.mean()
check2 = pos_mean > neg_mean
print(f"  [{'PASS' if check2 else 'WARN'}] Positive cells score higher: "
      f"pos_mean={pos_mean:.4f} > neg_mean={neg_mean:.4f}")
if not check2:
    print(f"    NOTE: Positive cells scoring lower than negative may indicate")
    print(f"    AHP weights are not well-calibrated for this sample area.")
    print(f"    This will be discussed in the AHP vs SHAP comparison.")

# Check 3 — Score variance is meaningful (not all cells same score)
score_std = features['ahp_score'].std()
check3 = score_std > 0.01
print(f"  [{'PASS' if check3 else 'FAIL'}] Score variance meaningful: "
      f"std={score_std:.4f}")
if not check3:
    checks_passed = False

# Check 4 — No NaN scores
nan_count = features['ahp_score'].isna().sum()
check4 = nan_count == 0
print(f"  [{'PASS' if check4 else 'FAIL'}] No NaN scores: "
      f"{nan_count} NaN values found")
if not check4:
    checks_passed = False

# Check 5 — CR was valid (already verified in Cell 5 — confirm here)
check5 = global_cr < 0.10
print(f"  [{'PASS' if check5 else 'FAIL'}] Consistency Ratio valid: "
      f"CR={global_cr:.4f} < 0.10")
if not check5:
    checks_passed = False

print()
if checks_passed:
    print("All validation checks passed. AHP scores are analytically valid.")
else:
    print("Some checks failed. Review before proceeding to Notebook 6.")

Running AHP score validation checks...

  [PASS] Score range in [0,1]: min=0.1043, max=0.8016
  [PASS] Positive cells score higher: pos_mean=0.5770 > neg_mean=0.4468
  [PASS] Score variance meaningful: std=0.1163
  [PASS] No NaN scores: 0 NaN values found
  [PASS] Consistency Ratio valid: CR=0.0175 < 0.10

All validation checks passed. AHP scores are analytically valid.


# Cell 10

In [ ]:
# Save AHP scores to Gold layer
# Columns saved: h3_id, ahp_score, ahp_rank, cluster scores,
# label, cafe_count, geometry

print("Saving AHP scores to Gold layer...")

cols_to_save = [
    'h3_id', 'label', 'cafe_count',
    'ahp_score', 'ahp_rank',
    'score_accessibility', 'score_demand',
    'score_urban', 'score_competition',
    'geometry'
]
cols_present = [c for c in cols_to_save if c in features.columns]
ahp_output   = features[cols_present].copy()

ahp_path = os.path.join(GOLD_PATH, 'gold_ahp_scores.geojson')
ahp_output.to_file(ahp_path, driver='GeoJSON')

print(f"  Saved: gold_ahp_scores.geojson ({len(ahp_output)} cells)")
print()
print("AHP score summary:")
print(ahp_output[['ahp_score', 'score_accessibility',
                   'score_demand', 'score_urban',
                   'score_competition']].describe().round(4).to_string())

Saving AHP scores to Gold layer...
  Saved: gold_ahp_scores.geojson (971 cells)

AHP score summary:
       ahp_score  score_accessibility  score_demand  score_urban  score_competition
count   971.0000             971.0000      971.0000     971.0000           971.0000
mean      0.4931               0.3195        0.6532       0.4496             0.8963
std       0.1163               0.1238        0.1646       0.2188             0.0963
min       0.1043               0.0000        0.1062       0.0000             0.0000
25%       0.4157               0.2416        0.5769       0.2897             0.8503
50%       0.4940               0.3064        0.7208       0.4371             0.9114
75%       0.5641               0.4008        0.7680       0.5712             0.9655
max       0.8016               0.8226        0.9226       1.0328             1.0000


# Cell 10B

In [ ]:
# =============================================================================
# TASK 3 — LEVEL 2 METRO SUB-WEIGHT SENSITIVITY TABLE
# =============================================================================
# The handover specifies: vary metro_count_norm sub-weight from 0.25 to 0.50
# in steps of 0.10, renormalise remaining Accessibility sub-weights
# proportionally. For each variation, recompute AHP effective weights and
# report whether the metro/transit divergence (AHP rank 5 → SHAP rank 13)
# holds or reverses.
#
# The SHAP ranking is fixed (from gold_shap_importance.csv, written by NB7).
# Since NB5 runs BEFORE NB7, the SHAP ranks are loaded from the file.
# If gold_shap_importance.csv does not yet exist (first full pipeline run),
# the cell falls back to the Phase 2 known ranks from the handover document.
#
# Requires (all defined by prior NB5 cells):
#   features          — GeoDataFrame with normalized columns
#   W                 — global cluster weights dict
#   ACCESS_SUB_WEIGHTS, DEMAND_SUB_WEIGHTS,
#   URBAN_SUB_WEIGHTS, COMPETITION_SUB_WEIGHTS
#   GOLD_PATH         — output directory path
# =============================================================================

print("=" * 65)
print("LEVEL 2 METRO SUB-WEIGHT SENSITIVITY TABLE (Task 3)")
print("=" * 65)
print("Varying metro_count_norm sub-weight: 0.25 → 0.50 (steps: 0.25, 0.35, 0.45, 0.50)")
print("Renormalising remaining Accessibility sub-weights proportionally.")
print()

# ── Fixed SHAP ranks (load from file if available, else use known Phase 2 values) ──
_shap_ranks_path = os.path.join(GOLD_PATH, 'gold_shap_importance.csv')
if os.path.exists(_shap_ranks_path):
    _shap_df = pd.read_csv(_shap_ranks_path)
    _shap_rank_map = dict(zip(_shap_df['feature_col'], _shap_df['shap_rank']))
    print(f"  SHAP ranks loaded from gold_shap_importance.csv")
else:
    # Fallback: Phase 2 known SHAP ranks from NB7 log (handover document)
    # Update these if NB7 is re-run with different data.
    _shap_rank_map = {
        'poi_diversity_norm':              1,
        'local_cafe_density_norm':         2,
        'competitor_saturation_inv_norm':  3,
        'retail_density_norm':             4,
        'tourist_poi_count_norm':          5,
        'pedestrian_street':               6,
        'pop_density_norm':                7,
        'night_light_norm':                8,
        'transit_density_norm':            9,
        'network_centrality_norm':        11,
        'office_density_norm':            12,
        'metro_count_norm':               13,
        'university_proximity_norm':      10,
    }
    print(f"  WARNING: gold_shap_importance.csv not found.")
    print(f"  Using Phase 2 known SHAP ranks from handover document.")
    print(f"  Re-run NB7 first to generate gold_shap_importance.csv for "
          f"exact values.")

# ── Metro sub-weight sweep ─────────────────────────────────────────────────────
# Canonical baseline sub-weights (mirrors Cell 6 / ACCESS_SUB_WEIGHTS)
_BASE_METRO      = ACCESS_SUB_WEIGHTS['metro_count_norm']        # 0.25
_BASE_TRANSIT    = ACCESS_SUB_WEIGHTS['transit_density_norm']    # 0.40
_BASE_CENTRALITY = ACCESS_SUB_WEIGHTS['network_centrality_norm'] # 0.35

_metro_sweep = [0.25, 0.35, 0.45, 0.50]   # as specified in handover

# All 10 AHP features (cluster order — mirrors Task 1 ordering)
_AHP_FEATS_NB5 = [
    'metro_count_norm', 'transit_density_norm', 'network_centrality_norm',  # Access
    'pop_density_norm', 'night_light_norm',                                   # Demand
    'retail_density_norm', 'poi_diversity_norm', 'pedestrian_street',
    'tourist_poi_count_norm',                                                  # Urban
    'competitor_saturation_inv_norm',                                          # Compet
]
_AHP_CLUSTER_W_NB5 = (
    [W['accessibility']]    * 3 +
    [W['demand_potential']] * 2 +
    [W['urban_context']]    * 4 +
    [W['competition']]      * 1
)

# Canonical sub-weights for non-Accessibility clusters (unchanged across sweep)
_NON_ACCESS_SW_NB5 = [
    # Demand
    DEMAND_SUB_WEIGHTS['pop_density_norm'],
    DEMAND_SUB_WEIGHTS['night_light_norm'],
    # Urban
    URBAN_SUB_WEIGHTS['retail_density_norm'],
    URBAN_SUB_WEIGHTS['poi_diversity_norm'],
    URBAN_SUB_WEIGHTS['pedestrian_street'],
    URBAN_SUB_WEIGHTS['tourist_poi_count_norm'],
    # Competition
    COMPETITION_SUB_WEIGHTS['competitor_saturation_inv_norm'],
]

# Feature matrix (all cells, 10 AHP features)
_X_ahp_nb5 = features[_AHP_FEATS_NB5].fillna(0).values  # shape: (n_cells, 10)

# ── Run sweep ─────────────────────────────────────────────────────────────────
rows = []
for metro_sw in _metro_sweep:
    # Renormalise remaining Accessibility sub-weights proportionally
    remaining_access = 1.0 - metro_sw
    old_remaining    = _BASE_TRANSIT + _BASE_CENTRALITY   # 0.75

    transit_sw    = _BASE_TRANSIT    * (remaining_access / old_remaining)
    centrality_sw = _BASE_CENTRALITY * (remaining_access / old_remaining)

    # Verify they still sum to 1.0 within Accessibility cluster
    access_sum = metro_sw + transit_sw + centrality_sw
    assert abs(access_sum - 1.0) < 1e-9, (
        f"Accessibility sub-weights do not sum to 1: {access_sum:.8f}"
    )

    # Effective global weights for all 10 AHP features under this scenario
    _access_sw_this = [metro_sw, transit_sw, centrality_sw]
    _all_sw_this    = _access_sw_this + _NON_ACCESS_SW_NB5

    eff_weights = np.array([
        cw * sw for cw, sw in zip(_AHP_CLUSTER_W_NB5, _all_sw_this)
    ])

    # AHP score under this sub-weight scenario
    ahp_perm_scores = _X_ahp_nb5 @ eff_weights

    # Rank all 10 AHP features by effective weight (descending, 1 = highest)
    # Build a Series for ranking
    eff_w_series = pd.Series(
        dict(zip(_AHP_FEATS_NB5, eff_weights))
    )
    eff_ranks = eff_w_series.rank(ascending=False, method='min').astype(int)

    # Key ranks for the metro-divergence finding
    metro_ahp_rank    = int(eff_ranks['metro_count_norm'])
    transit_ahp_rank  = int(eff_ranks['transit_density_norm'])
    central_ahp_rank  = int(eff_ranks['network_centrality_norm'])

    metro_shap_rank   = _shap_rank_map.get('metro_count_norm',   'N/A')
    transit_shap_rank = _shap_rank_map.get('transit_density_norm', 'N/A')

    # Delta for metro feature
    metro_delta = (
        abs(metro_ahp_rank - metro_shap_rank)
        if isinstance(metro_shap_rank, (int, float)) else 'N/A'
    )

    # Divergence direction for metro: AHP still ranks higher than SHAP?
    if isinstance(metro_shap_rank, (int, float)):
        metro_diverges = metro_ahp_rank < metro_shap_rank   # lower rank = higher priority
        direction_str  = "AHP > SHAP (divergence holds)" if metro_diverges else "SHAP ≥ AHP (divergence reversed)"
    else:
        direction_str = "N/A"

    rows.append({
        'metro_sub_w':       round(metro_sw, 2),
        'transit_sub_w':     round(transit_sw, 4),
        'centrality_sub_w':  round(centrality_sw, 4),
        'metro_eff_w':       round(float(eff_weights[0]), 4),
        'transit_eff_w':     round(float(eff_weights[1]), 4),
        'centrality_eff_w':  round(float(eff_weights[2]), 4),
        'metro_ahp_rank':    metro_ahp_rank,
        'transit_ahp_rank':  transit_ahp_rank,
        'centrality_ahp_rank': central_ahp_rank,
        'metro_shap_rank':   metro_shap_rank,
        'transit_shap_rank': transit_shap_rank,
        'metro_ahp_shap_delta': metro_delta,
        'metro_divergence_direction': direction_str,
        'is_baseline':       (abs(metro_sw - _BASE_METRO) < 1e-9),
    })

sens_df = pd.DataFrame(rows)

# ── Display table ─────────────────────────────────────────────────────────────
print("Sensitivity table — metro_count_norm sub-weight variation:")
print()
print(f"  {'metro_sw':<10} {'transit_sw':<12} {'central_sw':<12} "
      f"{'metro_eff_w':<13} "
      f"{'metro_AHP_rank':<16} {'metro_SHAP_rank':<17} "
      f"{'|Δ|':<5} Direction")
print(f"  {'-'*10} {'-'*12} {'-'*12} {'-'*13} "
      f"{'-'*16} {'-'*17} {'-'*5} {'-'*35}")

for _, row in sens_df.iterrows():
    baseline_tag = " ← BASELINE" if row['is_baseline'] else ""
    print(f"  {row['metro_sub_w']:<10.2f} "
          f"{row['transit_sub_w']:<12.4f} "
          f"{row['centrality_sub_w']:<12.4f} "
          f"{row['metro_eff_w']:<13.4f} "
          f"{row['metro_ahp_rank']:<16} "
          f"{row['metro_shap_rank']:<17} "
          f"{str(row['metro_ahp_shap_delta']):<5} "
          f"{row['metro_divergence_direction']}"
          f"{baseline_tag}")

print()

# ── Robustness conclusion ─────────────────────────────────────────────────────
all_diverge = all(
    (isinstance(r['metro_ahp_shap_delta'], (int, float)) and
     r['metro_ahp_rank'] < r['metro_shap_rank'])
    for _, r in sens_df.iterrows()
)

if all_diverge:
    print("ROBUSTNESS CHECK: Metro/transit AHP–SHAP divergence HOLDS across")
    print(f"all {len(_metro_sweep)} metro sub-weight scenarios tested "
          f"({min(_metro_sweep):.2f} – {max(_metro_sweep):.2f}).")
    print("The divergence finding is robust to Level 2 sub-weight variation.")
    print("Thesis language: 'The metro/transit divergence (AHP rank higher than")
    print("SHAP rank) persists across all tested Level 2 sub-weight scenarios,")
    print(f"confirming robustness to the specific choice of metro sub-weight.")
else:
    print("ROBUSTNESS CHECK: Metro/transit divergence REVERSES in at least one")
    print("sub-weight scenario — qualify the finding in the discussion chapter.")
    reversals = sens_df[sens_df['metro_ahp_rank'] >= sens_df['metro_shap_rank']]
    print(f"Reversal at metro sub-weights: "
          f"{list(reversals['metro_sub_w'].values)}")

# ── Save to Gold ──────────────────────────────────────────────────────────────
sens_path = os.path.join(GOLD_PATH, 'gold_ahp_metro_sensitivity.csv')
sens_df.to_csv(sens_path, index=False)
print()
print(f"Saved: gold_ahp_metro_sensitivity.csv ({len(sens_df)} rows)")

LEVEL 2 METRO SUB-WEIGHT SENSITIVITY TABLE (Task 3)
Varying metro_count_norm sub-weight: 0.25 → 0.50 (steps: 0.25, 0.35, 0.45, 0.50)
Renormalising remaining Accessibility sub-weights proportionally.

  SHAP ranks loaded from gold_shap_importance.csv
Sensitivity table — metro_count_norm sub-weight variation:

  metro_sw   transit_sw   central_sw   metro_eff_w   metro_AHP_rank   metro_SHAP_rank   |Δ|   Direction
  ---------- ------------ ------------ ------------- ---------------- ----------------- ----- -----------------------------------
  0.25       0.4000       0.3500       0.0941        5                13                8     AHP > SHAP (divergence holds) ← BASELINE
  0.35       0.3467       0.3033       0.1318        3                13                10    AHP > SHAP (divergence holds)
  0.45       0.2933       0.2567       0.1694        1                13                12    AHP > SHAP (divergence holds)
  0.50       0.2667       0.2333       0.1882        1                13 

# Cell 11

In [ ]:
log_lines = [
    f"Thesis — Phase 2 · AHP Scoring Log",
    f"====================================",
    f"Run date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}",
    f"",
    f"Feature counts (consistent with NB4 and thesis tables):",
    f"  15 GDF columns total   : 13 model features + viable_overlap_ratio + label",
    f"  13 RF model features   : all features passed to Random Forest",
    f"  10 AHP inputs (scoring) : 9 _norm columns + pedestrian_street (binary)",
    f"",
    f"Level 1 — Global cluster weights (4x4 pairwise matrix, CR={global_cr:.4f}):",
    f"  Accessibility:    {W['accessibility']:.4f} "
    f"({W['accessibility']*100:.1f}%)",
    f"  Demand Potential: {W['demand_potential']:.4f} "
    f"({W['demand_potential']*100:.1f}%)",
    f"  Urban Context:    {W['urban_context']:.4f} "
    f"({W['urban_context']*100:.1f}%)",
    f"  Competition:      {W['competition']:.4f} "
    f"({W['competition']*100:.1f}%)",
    f"  Consistency Ratio (CR): {global_cr:.4f} [VALID < 0.10]",
    f"",
    f"Level 2 — Sub-weights within clusters (directly assigned expert weights):",
    f"  Accessibility (ACCESS_SUB_WEIGHTS):",
    f"    metro_count_norm:               {ACCESS_SUB_WEIGHTS['metro_count_norm']:.2f}",
    f"    transit_density_norm:           {ACCESS_SUB_WEIGHTS['transit_density_norm']:.2f}",
    f"    network_centrality_norm:        {ACCESS_SUB_WEIGHTS['network_centrality_norm']:.2f}",
    f"  Demand Potential (DEMAND_SUB_WEIGHTS) — Phase 2 canonical 2-feature structure:",
    f"    pop_density_norm:               {DEMAND_SUB_WEIGHTS['pop_density_norm']:.2f}",
    f"    night_light_norm:               {DEMAND_SUB_WEIGHTS['night_light_norm']:.2f}",
    f"    [office_density and university_proximity excluded — RF-only per Phase 2 design]",
    f"  Urban Context (URBAN_SUB_WEIGHTS):",
    f"    retail_density_norm:            {URBAN_SUB_WEIGHTS['retail_density_norm']:.2f}",
    f"    poi_diversity_norm:             {URBAN_SUB_WEIGHTS['poi_diversity_norm']:.2f}",
    f"    pedestrian_street:              {URBAN_SUB_WEIGHTS['pedestrian_street']:.2f}",
    f"    tourist_poi_count_norm:         {URBAN_SUB_WEIGHTS['tourist_poi_count_norm']:.2f}",
    f"  Competition (COMPETITION_SUB_WEIGHTS):",
    f"    competitor_saturation_inv_norm: {COMPETITION_SUB_WEIGHTS['competitor_saturation_inv_norm']:.2f}",
    f"    [local_cafe_density excluded — RF-only feature, not AHP-scored]",
    f"",
    f"AHP score statistics:",
    f"  Mean:   {features['ahp_score'].mean():.4f}",
    f"  Std:    {features['ahp_score'].std():.4f}",
    f"  Min:    {features['ahp_score'].min():.4f}",
    f"  Max:    {features['ahp_score'].max():.4f}",
    f"",
    f"Score by class:",
    f"  Positive (label=1) mean: {pos_scores.mean():.4f}",
    f"  Negative (label=0) mean: {neg_scores.mean():.4f}",
    f"",
    f"Validation: {'ALL CHECKS PASSED' if checks_passed else 'SOME CHECKS FAILED'}",
    f"",
    f"Output: gold_ahp_scores.geojson",
]

log_path = os.path.join(GOLD_PATH, 'gold_ahp_log.txt')
with open(log_path, 'w') as f:
    f.write('\n'.join(log_lines))

print('\n'.join(log_lines))
print(f"\nLog saved to: {log_path}")

Thesis — Phase 2 · AHP Scoring Log
Run date: 2026-04-26 12:04

Feature counts (consistent with NB4 and thesis tables):
  15 GDF columns total   : 13 model features + viable_overlap_ratio + label
  13 RF model features   : all features passed to Random Forest
  10 AHP inputs (scoring) : 9 _norm columns + pedestrian_street (binary)

Level 1 — Global cluster weights (4x4 pairwise matrix, CR=0.0175):
  Accessibility:    0.3764 (37.6%)
  Demand Potential: 0.2998 (30.0%)
  Urban Context:    0.2535 (25.3%)
  Competition:      0.0703 (7.0%)
  Consistency Ratio (CR): 0.0175 [VALID < 0.10]

Level 2 — Sub-weights within clusters (directly assigned expert weights):
  Accessibility (ACCESS_SUB_WEIGHTS):
    metro_count_norm:               0.25
    transit_density_norm:           0.40
    network_centrality_norm:        0.35
  Demand Potential (DEMAND_SUB_WEIGHTS) — Phase 2 canonical 2-feature structure:
    pop_density_norm:               0.45
    night_light_norm:               0.55
    [office_de

# Cell 12

In [ ]:
print("=" * 60)
print("NOTEBOOK 5 COMPLETE — AHP Scoring Summary")
print("=" * 60)

expected_files = [
    'gold_ahp_scores.geojson',
    'gold_ahp_metro_sensitivity.csv',
    'gold_ahp_log.txt'
]

all_present = True
for filename in expected_files:
    filepath = os.path.join(GOLD_PATH, filename)
    status   = "OK" if os.path.exists(filepath) else "MISSING"
    if status == "MISSING":
        all_present = False
    print(f"  [{status}] {filename}")

print()
print(f"  Global CR:            {global_cr:.4f} [valid < 0.10]")
print(f"  Cells scored:         {len(features)}")
print(f"  AHP score mean:       {features['ahp_score'].mean():.4f}")
print(f"  Positive class mean:  {pos_scores.mean():.4f}")
print(f"  Negative class mean:  {neg_scores.mean():.4f}")

print()
if all_present:
    print("All outputs present. Ready to proceed to Notebook 6.")
    print("Next: Random Forest training, validation, and SHAP computation.")
else:
    print("Some files missing. Review cells above before proceeding.")

NOTEBOOK 5 COMPLETE — AHP Scoring Summary
  [OK] gold_ahp_scores.geojson
  [OK] gold_ahp_metro_sensitivity.csv
  [OK] gold_ahp_log.txt

  Global CR:            0.0175 [valid < 0.10]
  Cells scored:         971
  AHP score mean:       0.4931
  Positive class mean:  0.5770
  Negative class mean:  0.4468

All outputs present. Ready to proceed to Notebook 6.
Next: Random Forest training, validation, and SHAP computation.
